Сперва установим необходимые библиотеки

In [ ]:
!pip install datasets==2.14.6
!pip install huggingface-hub==0.20.0
!pip install evaluate transformers rouge-score nltk >> None
!apt install git-lfs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 493.7/493.7 kB 13.3 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 2.16.0
    Uninstalling datasets-2.16.0:
      Successfully uninstalled datasets-2.16.0
  Using cached huggingface_hub-0.20.0-py3-none-any.whl.metadata (12 kB)
Using cached huggingface_hub-0.20.0-py3-none-any.whl (329 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.33.4
    Uninstalling huggingface-hub-0.33.4:
      Successfully uninstalled huggingface-hub-0.33.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.51.3 requires huggingface-hub<1.0,>=0.30.0, but you have huggingface-hub 0.20.0 which is incompatible.
diffusers 0.34.0 requires huggingface-hub>=0.27.0, but you have huggingface-hub 0.20.0 which is incompatible.
gradio 5.31.0 requires huggi

Логинимся в hugging face через аутентификационный токен

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
import transformers

print(transformers.__version__)

4.51.3


In [ ]:
import datasets
print(datasets.__version__)

2.14.6


# Fine-tuning модели на задаче суммаризации

В этом ноутбуке используем модель [`t5-small`](https://huggingface.co/t5-small) на датасете [XSum dataset](https://arxiv.org/pdf/1808.08745.pdf) (extreme summarization) новости и их саммари в 1 предложении.
:

In [ ]:
model_checkpoint = "t5-small" # выбираем модель

## Загружаем и исследуем датасет

In [ ]:
from datasets import load_dataset, DownloadMode
from evaluate import load

import os

# Clear the datasets cache
os.system("rm -rf ~/.cache/huggingface/datasets/")

raw_datasets = load_dataset("EdinburghNLP/xsum")
metric = load("rouge")

Generating train split:   0%|          | 0/204045 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11332 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11334 [00:00<?, ? examples/s]

In [ ]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 204045
    })
    validation: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 11332
    })
    test: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 11334
    })
})

Выведем пример текста и саммари

In [ ]:
raw_datasets["train"][0]

{'document': 'The full cost of damage in Newton Stewart, one of the areas worst affected, is still being assessed.\nRepair work is ongoing in Hawick and many roads in Peeblesshire remain badly affected by standing water.\nTrains on the west coast mainline face disruption due to damage at the Lamington Viaduct.\nMany businesses and householders were affected by flooding in Newton Stewart after the River Cree overflowed into the town.\nFirst Minister Nicola Sturgeon visited the area to inspect the damage.\nThe waters breached a retaining wall, flooding many commercial properties on Victoria Street - the main shopping thoroughfare.\nJeanette Tate, who owns the Cinnamon Cafe which was badly affected, said she could not fault the multi-agency response once the flood hit.\nHowever, she said more preventative work could have been carried out to ensure the retaining wall did not fail.\n"It is difficult but I do think there is so much publicity for Dumfries and the Nith - and I totally apprecia

In [ ]:
import datasets
import random
import pandas as pd
from IPython.display import display, HTML

## Предобработка данных



Используем токенизатор, который идет вместе с моделью

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [ ]:
tokenizer("Hello, this is one sentence!")

{'input_ids': [8774, 6, 48, 19, 80, 7142, 55, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}

Задаем функцию, для обработки текстов в последовательность токенов

In [ ]:
max_input_length = 1024
max_target_length = 128

prefix = "summarize: "

def preprocess_function(examples):
    inputs = [prefix + doc for doc in examples["document"]]
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)

    # Setup the tokenizer for targets
    labels = tokenizer(text_target=examples["summary"], max_length=max_target_length, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

Пример того, как выглядят данные после предобработки

In [ ]:
preprocess_function(raw_datasets['train'][:2])

{'input_ids': [[21603, 10, 37, 423, 583, 13, 1783, 16, 20126, 16496, 6, 80, 13, 8, 844, 6025, 4161, 6, 19, 341, 271, 14841, 5, 7057, 161, 19, 4912, 16, 1626, 5981, 11, 186, 7540, 16, 1276, 15, 2296, 7, 5718, 2367, 14621, 4161, 57, 4125, 387, 5, 15059, 7, 30, 8, 4653, 4939, 711, 747, 522, 17879, 788, 12, 1783, 44, 8, 15763, 6029, 1813, 9, 7472, 5, 1404, 1623, 11, 5699, 277, 130, 4161, 57, 18368, 16, 20126, 16496, 227, 8, 2473, 5895, 15, 147, 89, 22411, 139, 8, 1511, 5, 1485, 3271, 3, 21926, 9, 472, 19623, 5251, 8, 616, 12, 15614, 8, 1783, 5, 37, 13818, 10564, 15, 26, 3, 9, 3, 19513, 1481, 6, 18368, 186, 1328, 2605, 30, 7488, 1887, 3, 18, 8, 711, 2309, 9517, 89, 355, 5, 3966, 1954, 9233, 15, 6, 113, 293, 7, 8, 16548, 13363, 106, 14022, 84, 47, 14621, 4161, 6, 243, 255, 228, 59, 7828, 8, 1249, 18, 545, 11298, 1773, 728, 8, 8347, 1560, 5, 611, 6, 255, 243, 72, 1709, 1528, 161, 228, 43, 118, 4006, 91, 12, 766, 8, 3, 19513, 1481, 410, 59, 5124, 5, 96, 196, 17, 19, 1256, 68, 27, 103, 317, 132

**Уменьшаем размер датасета**

В Google Colab существуют ограничения на бесплатные вычислительные ресурсы, которые могут меняться со временем. Поэтому большой датасет, используемый экспертом, скорее всего не удастся полностью обработать в рамках Colab. Чтобы ускорить обучение и избежать сбоев, мы уменьшаем размер датасета.

In [ ]:
raw_datasets["train"] = raw_datasets["train"].shuffle(seed=42).select(range(5000))
raw_datasets["validation"] = raw_datasets["validation"].shuffle(seed=42).select(range(1000))

Таким образом, для дообучения будет использоваться только первые 5000 строк тренировочного набора и 1000 строк валидационного.

In [ ]:
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/11334 [00:00<?, ? examples/s]

## Дообучаем модель

Т.к. исходная и новая модели работают в режиме Sequence-to-Sequence, используем `AutoModelForSeq2SeqLM`

In [ ]:
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Задаем гиперпараметры обучения:

In [ ]:
batch_size = 16
model_name = model_checkpoint.split("/")[-1]
args = Seq2SeqTrainingArguments(
    output_dir=f"{model_name}-finetuned-xsum",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True,
    push_to_hub=False, # Do not publish model on HF
    report_to=[] # Do not use external experiment tracking tools like wandb or tensorboard
)

Задаем упаковщик данных для того, чтобы упаковывать примеры в батчи для параллельного обучения

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

Используем метрику Rouge, а так же будем замерять длину сгенерированных саммари

In [ ]:
import nltk
import numpy as np

nltk.download('punkt')

nltk.data.path.append('/root/nltk_data')

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    # Replace -100 in the labels as we can't decode them.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Rouge expects a newline after each sentence
    decoded_preds = ["\n".join(nltk.sent_tokenize(pred.strip())) for pred in decoded_preds]
    decoded_labels = ["\n".join(nltk.sent_tokenize(label.strip())) for label in decoded_labels]

    # Note that other metrics may not have a `use_aggregator` parameter
    # and thus will return a list, computing a metric for each sentence.
    result = metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True, use_aggregator=True)
    # Extract a few results
    result = {key: value * 100 for key, value in result.items()}

    # Add mean generated length
    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)

    return {k: round(v, 4) for k, v in result.items()}

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Передаем все гиперпараметры в Trainer:

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

И запускаем обучение:


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,No log,2.709840,23.184600,4.804100,18.012700,18.042500,19.579000
2,2.947200,2.682102,24.716300,5.512800,19.136500,19.160700,19.625000
3,2.947200,2.676170,24.984400,5.704400,19.421000,19.438900,19.637000


TrainOutput(global_step=939, training_loss=2.920869057424787, metrics={'train_runtime': 940.3709, 'train_samples_per_second': 15.951, 'train_steps_per_second': 0.999, 'total_flos': 3968276601765888.0, 'train_loss': 2.920869057424787, 'epoch': 3.0})

Теперь посмотрим как работает суммаризация на тестовой выборке - тех текстах в датасете которые мы не использовали для обучения

In [ ]:
from transformers import pipeline

# Create a summarization pipeline with your trained model
summarizer = pipeline(
    "summarization",
    model=model,
    tokenizer=tokenizer,
)

# Test on validation examples
for i in range(5):  # Test first 3 examples
    document = tokenized_datasets["test"][i]["document"]
    reference = tokenized_datasets["test"][i]["summary"]

    # Generate summary
    generated = summarizer(
        document,
        max_length=50,  # Adjust based on your needs
        min_length=10,
        do_sample=False,  # Use beam search for better quality
        num_beams=4
    )

    print(f"Example {i+1}:")
    print(f"Document:\n{document[:300]}...")
    print(f"Reference:\n{reference}")
    print(f"Generated:\n{generated[0]['summary_text']}")
    print("-" * 50)

Device set to use cuda:0


Example 1:
Document:
Prison Link Cymru had 1,099 referrals in 2015-16 and said some ex-offenders were living rough for up to a year before finding suitable accommodation.
Workers at the charity claim investment in housing would be cheaper than jailing homeless repeat offenders.
The Welsh Government said more people than...
Reference:
There is a "chronic" need for more housing for prison leavers in Wales, according to a charity.
Generated:
, a homeless charity, has said it is "chronic". The Welsh Government has said more people than ever were getting help to address housing problems.
--------------------------------------------------
Example 2:
Document:
Officers searched properties in the Waterfront Park and Colonsay View areas of the city on Wednesday.
Detectives said three firearms, ammunition and a five-figure sum of money were recovered.
A 26-year-old man who was arrested and charged appeared at Edinburgh Sheriff Court on Thursday....
Reference:
A man has appeared in court after fi